# Homework Starter — Stage 05: Data Storage
Name: Pei Syuan Lin
Date: 2026-08-24 

Objectives:
- Env-driven paths to `data/raw/` and `data/processed/`
- Save CSV and Parquet; reload and validate
- Abstract IO with utility functions; document choices

In [6]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install numpy
# !pip install pandas
# %pip install pyarrow
# !pip install python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.0/36.0 MB 39.5 MB/s  0:00:00 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [1]:
# --- files this notebook needs (run me first - I only report, I change nothing) ---
from pathlib import Path

ROOT = Path.cwd()          # notebooks are meant to be run from their own folder
CHECKS = [
    (".env", "NEEDED", "YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing"),
    (".env.example", "NEEDED", "shipped with this stage - the template you copy to .env"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    print(f"\n{missing} needed file(s) missing. Put them at the paths above, relative to:\n  {ROOT}")
    print("If that folder looks wrong, you are running the notebook from the wrong place.")
else:
    print("\nAll needed files present.")

Looking in: /Users/cloudnine_7/bootcamp_elrie_lin/homework/homework05

  [OK ]  NEEDED    .env                                YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing
  [OK ]  NEEDED    .env.example                        shipped with this stage - the template you copy to .env

All needed files present.


In [2]:
import os, pathlib, datetime as dt
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
RAW = pathlib.Path(os.getenv('DATA_DIR_RAW', 'data/raw'))
PROC = pathlib.Path(os.getenv('DATA_DIR_PROCESSED', 'data/processed'))
RAW.mkdir(parents=True, exist_ok=True)
PROC.mkdir(parents=True, exist_ok=True)
print('RAW ->', RAW.resolve())
print('PROC ->', PROC.resolve())

RAW -> /Users/cloudnine_7/bootcamp_elrie_lin/homework/homework05/data/raw
PROC -> /Users/cloudnine_7/bootcamp_elrie_lin/homework/homework05/data/processed


## 1) Create or Load a Sample DataFrame
You may reuse data from prior stages or create a small synthetic dataset.

In [3]:
import numpy as np

# Generate reproducible synthetic time series dataset
np.random.seed(42)

dates = pd.date_range('2024-01-01', periods=20, freq='D')
df = pd.DataFrame({'date': dates, 'ticker': ['AAPL']*20, 'price': 150 + np.random.randn(20).cumsum()})
df.head()

,date,ticker,price
0,2024-01-01,AAPL,150.496714
1,2024-01-02,AAPL,150.358450
2,2024-01-03,AAPL,151.006138
3,2024-01-04,AAPL,152.529168
4,2024-01-05,AAPL,152.295015


## 2) Save CSV to data/raw/ and Parquet to data/processed/ (TODO)
- Use timestamped filenames.
- Handle missing Parquet engine gracefully.

In [9]:
def ts(): return dt.datetime.now().strftime('%Y%m%d-%H%M%S')

# TODO: Save CSV
csv_path = RAW / f"sample_{ts()}.csv"
df.to_csv(csv_path, index=False)
print("Saved CSV to:", csv_path)

# TODO: Save Parquet
pq_path = RAW / f"sample_{ts()}.parquet"
try:
    df.to_parquet(pq_path)
    print("Saved Parquet to:", pq_path)
except Exception as e:
    print('Parquet engine not available. Install pyarrow or fastparquet to complete this step.')
    print('Error detail:', e)
    pq_path = None

Saved CSV to: data/raw/sample_20260824-165824.csv
Saved Parquet to: data/raw/sample_20260824-165824.parquet


## 3) Reload and Validate (TODO)
- Compare shapes and key dtypes.

In [10]:
def validate_loaded(original, reloaded):
    checks = {
        'shape_equal': original.shape == reloaded.shape,
        'date_is_datetime': pd.api.types.is_datetime64_any_dtype(reloaded['date']) if 'date' in reloaded.columns else False,
        'price_is_numeric': pd.api.types.is_numeric_dtype(reloaded['price']) if 'price' in reloaded.columns else False,
    }
    return checks

df_csv = pd.read_csv(csv_path, parse_dates=['date'])
validate_loaded(df, df_csv)

{'shape_equal': True, 'date_is_datetime': True, 'price_is_numeric': True}

In [12]:
if pq_path:
    try:
        df_pq = pd.read_parquet(pq_path)
        v_pq = validate_loaded(df, df_pq)
        print("Parquet Reload Validation Results:", v_pq)
    except Exception as e:
        print('Parquet read failed:', e)
else:
    print("Skipping Parquet validation since file was not created.")        

Parquet Reload Validation Results: {'shape_equal': True, 'date_is_datetime': True, 'price_is_numeric': True}


## 4) Utilities (TODO)
- Implement `detect_format`, `write_df`, `read_df`.
- Use suffix to route; create parent dirs if needed; friendly errors for Parquet.

In [14]:
import typing as t, pathlib

def detect_format(path: t.Union[str, pathlib.Path]):
    s = str(path).lower()
    if s.endswith('.csv'): return 'csv'
    if s.endswith('.parquet') or s.endswith('.pq') or s.endswith('.parq'): return 'parquet'
    raise ValueError('Unsupported format: ' + s)

def write_df(df: pd.DataFrame, path: t.Union[str, pathlib.Path]):
    p = pathlib.Path(path); p.parent.mkdir(parents=True, exist_ok=True)
    fmt = detect_format(p)
    if fmt == 'csv':
        df.to_csv(p, index=False)
    else:
        try:
            df.to_parquet(p)
        except Exception as e:
            raise RuntimeError('Parquet engine not available. Install pyarrow or fastparquet.') from e
    return p

def read_df(path: t.Union[str, pathlib.Path]):
    p = pathlib.Path(path)
    fmt = detect_format(p)
    if fmt == 'csv':
        return pd.read_csv(p, parse_dates=['date']) if 'date' in pd.read_csv(p, nrows=0).columns else pd.read_csv(p)
    else:
        try:
            return pd.read_parquet(p)
        except Exception as e:
            raise RuntimeError('Parquet engine not available. Install pyarrow or fastparquet.') from e

# Demo
p_csv = RAW / f"util_{ts()}.csv"
p_pq  = RAW / f"util_{ts()}.parquet"

# Demo CSV utility write & read
write_df(df, p_csv)
df_util_csv = read_df(p_csv)
print("Utility CSV read preview:")
print(df_util_csv.head(2))

# Demo Parquet utility write & read
try:
    write_df(df, p_pq)
    df_util_pq = read_df(p_pq)
    print("\nUtility Parquet read preview:")
    print(df_util_pq.head(2))
except RuntimeError as e:
    print('Skipping Parquet util demo:', e)

Utility CSV read preview:
        date ticker       price
0 2024-01-01   AAPL  150.496714
1 2024-01-02   AAPL  150.358450

Utility Parquet read preview:
        date ticker       price
0 2024-01-01   AAPL  150.496714
1 2024-01-02   AAPL  150.358450


## 5) Documentation

### Data Storage Protocol
- **Folder Organization**:
  - `data/raw/`: Dedicated to immutable, raw CSV snapshots fetched from API/Scraping steps.
  - `data/processed/`: Dedicated to optimized, binary Parquet files used for downstream EDA and analytical modeling.
- **Format Rationale**:
  - **CSV**: Human-readable, easy inspection for raw data ingestion.
  - **Parquet**: Columnar format with built-in schema, compressed storage size, and native preservation of Data Types (e.g. `datetime64`).
- **Environment Driven**:
  - Paths are dynamically resolved via `.env` environment variables (`DATA_DIR_RAW` and `DATA_DIR_PROCESSED`).
  - Ensures portability across local development machines and production pipeline environments.

### Validation Summary
- `shape_equal`: Confirmed exact row and column dimensions between source and reloaded datasets.
- `dtype_preservation`: Verified native preservation of `datetime64` types in Parquet and auto-parsed datetime types in CSV.